# W1-T02a EDA — PDB RNA chain parse

Reads `data/pdb_rna/metadata.parquet` (scalars) and the per-chain pkls
(for the modified-nt distribution, which is nested and lives only in pkl).

Acceptance items covered here:
- length / resolution / release-year distributions
- modified-nt top-20
- split + num_models_source breakdown (sanity for the num_models decision)

In [ ]:
import pickle
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path('../data/pdb_rna/raw')
META = Path('../data/pdb_rna/metadata.parquet')
df = pd.read_parquet(META)
print(df.shape)
df.head()

In [ ]:
# Split & num_models provenance — confirms the num_models policy
print(df['in_rna3db_split'].value_counts(dropna=False))
print()
print(df['num_models_source'].value_counts(dropna=False))
print()
print('chains with num_models=None (NMR, deferred):',
      df['num_models'].isna().sum())

In [ ]:
# Length distribution
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df['length'].plot.hist(bins=80, ax=ax[0])
ax[0].set_title('Chain length'); ax[0].set_xlabel('nt')
df[df['length'] < 600]['length'].plot.hist(bins=60, ax=ax[1])
ax[1].set_title('Chain length (<600 nt zoom)'); ax[1].set_xlabel('nt')
plt.tight_layout(); plt.show()
df['length'].describe()

In [ ]:
# Resolution distribution (None = NMR/unknown, excluded from hist)
res = df['resolution'].dropna()
res.plot.hist(bins=60, figsize=(8, 4))
plt.title(f'Resolution (n={len(res)}, {df["resolution"].isna().sum()} None)')
plt.xlabel('Angstrom'); plt.show()
res.describe()

In [ ]:
# Release year distribution
yr = pd.to_datetime(df['release_date'], errors='coerce').dt.year
yr.value_counts().sort_index().plot.bar(figsize=(13, 4))
plt.title('Chains by release year'); plt.xlabel('year'); plt.show()

In [ ]:
# Modified-nt top 20 — nested data, read from pkls
c = Counter()
for p in RAW.glob('*.pkl'):
    d = pickle.load(open(p, 'rb'))
    for _pos, orig, _mapped in d['modified_residues']:
        c[orig] += 1
top = pd.Series(dict(c.most_common(20)))
top.plot.bar(figsize=(13, 4))
plt.title('Top-20 modified nucleotides (3-letter CCD code)')
plt.ylabel('occurrences'); plt.show()
top

In [ ]:
# Resolved-fraction sanity: SEQRES length vs residues with coordinates.
# Chains with low resolved fraction will have sparse atom_mask in T02b.
frac = df['n_resolved_residues'] / df['length']
frac.plot.hist(bins=50, figsize=(8, 4))
plt.title('Resolved fraction (n_resolved / SEQRES length)')
plt.xlabel('fraction'); plt.show()
print('chains <50% resolved:', (frac < 0.5).sum())
print('chains 100% resolved:', (frac == 1.0).sum())

In [ ]:
# Warnings audit — any chain with a parse_warning needs a human look
warned = []
for p in RAW.glob('*.pkl'):
    d = pickle.load(open(p, 'rb'))
    if d['parse_warnings']:
        warned.append((p.name, d['parse_warnings']))
print(f'{len(warned)} chains with warnings')
for n, w in warned[:20]:
    print(n, w)